# NCBI Disease NER: domain adaptation (continued MLM pre-training) on Colab

Direct fine-tuning of `bert` (bert-base-cased) vs. continued masked-LM pre-training on unlabelled PubMed
abstracts (DAPT) followed by the same fine-tuning (`ner.train`, seeds 1/2/3, FP32, dev selection by strict F1).

Runtime → Change runtime type → **GPU**. Everything that takes long is written to Google Drive:
the corpus, the DAPT model and the NER runs. `ner.sweep` skips runs whose `metrics.json` exists, so after
a disconnect re-run all cells (the DAPT cell is skipped once its `dapt.json` exists; DAPT itself cannot
resume mid-run).

In [ ]:
# ---- settings ----
BRANCH = 'domain-adaptation'
N_DOCS = 30000          # first N PMIDs of dapt/pmids.txt (year-balanced)
DAPT_STEPS = 10000      # optimizer steps of continued pre-training (set after the timing probe)
DAPT_BATCH = 32
DAPT_MAXLEN = 256
DAPT_PRECISION = 'fp32' # a T4 has no native bf16; 'fp16' is faster but deviates from the FP32 protocol
SEEDS = '1 2 3'

DRIVE = '/content/drive/MyDrive/comp9444_ncbi_dapt'
CORPUS_DIR = f'{DRIVE}/corpus_{N_DOCS}'
DAPT_DIR = f'{DRIVE}/bert_dapt_{N_DOCS}docs_{DAPT_STEPS}steps'
DAPT0_DIR = f'{DRIVE}/bert_dapt0'
RUNS_DIR = f'{DRIVE}/runs_v2_fp32_strict'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p {DRIVE} {RUNS_DIR}

In [ ]:
!rm -rf /content/group_project
!git clone -q -b {BRANCH} https://github.com/lofty14/bert-ner-baseline.git /content/group_project
%cd /content/group_project
!git log --oneline -1

In [ ]:
# keep Colab's preinstalled torch
!pip install -q --use-pep517 "transformers==5.22.0" "accelerate==1.15.0" "seqeval==1.2.2"
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.is_available() and torch.cuda.get_device_name(0))

## 1. Corpus (efetch + two-pass overlap removal)

Cached in `raw.jsonl` on Drive, so a re-run only fetches what is missing. Read the audit summary:
pass 1 removes NCBI Disease PMIDs, pass 2 removes documents containing a dev/test sentence.

In [ ]:
!python -m ner.dapt_corpus build --limit {N_DOCS} --out_dir {CORPUS_DIR} 2>&1 | grep -v "fetched "
!python -m ner.dapt_corpus positive_control   # the text-overlap check must fire on known overlap

## 2. Timing probe (200 steps)

Prints seconds per step and extrapolates one epoch over the whole corpus; use it to set `DAPT_STEPS`.

In [ ]:
import json, math
!python -m ner.dapt --corpus_dir {CORPUS_DIR} --max_steps 200 --eval_steps 100000 --batch_size {DAPT_BATCH} \
    --max_length {DAPT_MAXLEN} --precision {DAPT_PRECISION} --output_dir /content/dapt_probe 2>&1 | grep -E "^  |^=="
r = json.load(open('/content/dapt_probe/dapt.json'))
s = r['seconds']['per_step']
epoch_steps = math.ceil(r['corpus']['windows']['train'] / DAPT_BATCH)
print(f"{s:.3f} s/step; one epoch = {epoch_steps} steps ≈ {epoch_steps * s / 3600:.2f} h; "
      f"DAPT_STEPS={DAPT_STEPS} ≈ {DAPT_STEPS * s / 3600:.2f} h (+ evaluation)")

## 3. Continued pre-training

In [ ]:
import os
if os.path.exists(f'{DAPT_DIR}/dapt.json'):
    print('exists, skipped:', DAPT_DIR)
else:
    !python -m ner.dapt --corpus_dir {CORPUS_DIR} --max_steps {DAPT_STEPS} --batch_size {DAPT_BATCH} \
        --max_length {DAPT_MAXLEN} --precision {DAPT_PRECISION} --eval_steps 1000 --output_dir {DAPT_DIR}
r = json.load(open(f'{DAPT_DIR}/dapt.json'))
print(r['steps'], 'steps; valid loss', r['valid_loss'], 'ppl', r['valid_perplexity'])

In [ ]:
# optional zero-step control: same code path, original weights (should match direct fine-tuning)
if not os.path.exists(f'{DAPT0_DIR}/dapt.json'):
    !python -m ner.dapt --corpus_dir {CORPUS_DIR} --max_steps 0 --max_length {DAPT_MAXLEN} --output_dir {DAPT0_DIR}

## 4. NER fine-tuning, both arms (identical `ner.train` settings)

In [ ]:
!python -m ner.sweep --models bert --seeds {SEEDS} --runs_dir {RUNS_DIR}
!python -m ner.sweep --models bert --seeds {SEEDS} --runs_dir {RUNS_DIR} --init_from {DAPT_DIR}/model --label bert+dapt

In [ ]:
# optional: zero-step control arm
!python -m ner.sweep --models bert --seeds {SEEDS} --runs_dir {RUNS_DIR} --init_from {DAPT0_DIR}/model --label bert+dapt0

## 5. Results

In [ ]:
!python -m ner.aggregate {RUNS_DIR} --out {RUNS_DIR}/results.md
!python -m ner.compare {RUNS_DIR} --a bert --b bert+dapt --out {RUNS_DIR}/compare_dapt.md
!cat {RUNS_DIR}/compare_dapt.md